# ClaimCheck: Streamlit App (Google Colab)
**Qwen2.5-3B-Instruct + QLoRA adapter · Streamlit UI · public link via Cloudflare tunnel**

Veltris × Woxsen University · SLM Assignment · Shaik Shazia Mahek

**How to use:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.
Upload the adapter zip when asked. The last cell prints your public link.
The link works while this Colab tab is running.

In [ ]:
# ── Step 1: Install ─────────────────────────────────────────
!pip install -q "transformers>=4.46.0,<5.0.0" "peft>=0.13.0,<1.0.0" "bitsandbytes>=0.44.0" \
    "accelerate>=1.0.0" streamlit pytesseract Pillow PyMuPDF pandas
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print("✅ Installed. GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ── Step 2: Upload adapter zip + pre-download Qwen ─────────────
import os, glob, zipfile
from google.colab import files

if not glob.glob("./claimcheck-adapter/**/adapter_config.json", recursive=True):
    print("📤 Upload claimcheck-qwen2.5-3b-lora-adapter.zip")
    up = files.upload()
    with zipfile.ZipFile(list(up.keys())[0]) as z:
        z.extractall("./claimcheck-adapter")

ADAPTER_DIR = os.path.abspath(os.path.dirname(
    glob.glob("./claimcheck-adapter/**/adapter_config.json", recursive=True)[0]))
os.environ["ADAPTER_DIR"] = ADAPTER_DIR
print("✅ Adapter:", ADAPTER_DIR, os.listdir(ADAPTER_DIR))

# Download base model now so the app opens faster later
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen2.5-3B-Instruct")
print("✅ Qwen2.5-3B-Instruct downloaded")

In [ ]:
%%writefile pipeline.py
"""ClaimCheck core pipeline: read documents, extract fields with the fine-tuned SLM,
compare them (Python rules + SLM meaning check), chat about the case, audit log.
The AI never approves or rejects. A human reviewer makes the final decision."""
import os, re, json, hashlib, datetime

CANONICAL_FIELDS = ["patient_name", "age", "gender", "hospital_name", "admission_date",
                    "discharge_date", "diagnosis", "procedure", "total_amount"]
MEANING_FIELDS = ("diagnosis", "procedure", "hospital_name")   # SLM checks meaning here
AUDIT_FILE = os.environ.get("AUDIT_FILE", "audit_log.jsonl")

# Must match the training system prompt exactly
SYSTEM_PROMPT = """You are ClaimCheck, a medical document field extractor for Indian hospital claims.

Given a hospital document (claim form or discharge summary), extract these fields with EXACT quotes from the text:
- patient_name, age, gender, hospital_name, admission_date, discharge_date, diagnosis, procedure, total_amount

For EACH field, provide:
1. "value": The extracted value (cleaned up)
2. "quote": The EXACT text span from the document where you found this

Output valid JSON only. If a field is not found, set value to null and quote to null."""

MODEL = None
TOKENIZER = None


# ───────────────────────── Model ─────────────────────────
def load_model(adapter_dir, base_id="Qwen/Qwen2.5-3B-Instruct"):
    """Load Qwen2.5-3B in 4-bit + the ClaimCheck LoRA adapter (needs a GPU)."""
    global MODEL, TOKENIZER
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    TOKENIZER = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
    if TOKENIZER.pad_token is None:
        TOKENIZER.pad_token = TOKENIZER.eos_token
    base = AutoModelForCausalLM.from_pretrained(base_id, quantization_config=bnb,
                                                device_map="auto", trust_remote_code=True,
                                                torch_dtype=torch.float16,
                                                attn_implementation="eager")
    MODEL = PeftModel.from_pretrained(base, adapter_dir)
    MODEL.eval()
    return MODEL, TOKENIZER


def _generate(messages, max_new_tokens=512, use_adapter=True):
    """use_adapter=True  -> fine-tuned extractor
       use_adapter=False -> plain Qwen (better for chat and meaning checks)"""
    import torch, contextlib
    text = TOKENIZER.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = TOKENIZER(text, return_tensors="pt").to(MODEL.device)
    ctx = contextlib.nullcontext() if use_adapter else MODEL.disable_adapter()
    with torch.no_grad(), ctx:
        out = MODEL.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=TOKENIZER.pad_token_id)
    return TOKENIZER.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


# ───────────────────────── 1. Read documents ─────────────────────────
def read_document(file_path):
    """Return (text, method). Handles plain text, PDF (text or scanned) and images."""
    if not file_path or not os.path.exists(file_path):
        return None, "file not found"
    with open(file_path, "rb") as f:
        header = f.read(8)
    ext = os.path.splitext(file_path)[1].lower()
    try:
        if header[:5] == b"%PDF-" or ext == ".pdf":
            return _read_pdf(file_path)
        if (header[:8] == b"\x89PNG\r\n\x1a\n" or header[:2] == b"\xff\xd8"
                or ext in (".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".webp")):
            return _read_image(file_path)
    except Exception as e:
        return None, f"could not read: {e}"
    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        text = f.read()
    return (text, "text") if text.strip() else (None, "file is empty")


def _read_pdf(path):
    import fitz, pytesseract
    from PIL import Image
    doc, parts, used_ocr = fitz.open(path), [], False
    for page in doc:
        t = page.get_text().strip()
        if len(t) > 50:
            parts.append(t)
        else:
            pix = page.get_pixmap(dpi=300)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            parts.append(_ocr(img))
            used_ocr = True
    doc.close()
    return "\n\n".join(parts), ("pdf+ocr" if used_ocr else "pdf")


def _ocr(img):
    """OCR tuned for forms: grayscale + autocontrast, and page mode 6 (one uniform block),
    which keeps table cells such as 'Age / Sex: 52 yrs / F' that the default mode drops."""
    import pytesseract
    from PIL import ImageOps
    return pytesseract.image_to_string(ImageOps.autocontrast(img.convert("L")), config="--psm 6")


def _read_image(path):
    from PIL import Image
    return _ocr(Image.open(path)), "ocr"


# ───────────────────────── 2. Extract fields (fine-tuned SLM) ─────────────────────────
def extract_fields(text, doc_type_label):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Extract all fields from this {doc_type_label}:\n\n{text}"}]
    raw = _generate(messages, max_new_tokens=512, use_adapter=True)
    return parse_model_json(raw), raw


def parse_model_json(text):
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        return None
    for candidate in (m.group(), repair_json(m.group())):
        try:
            return json.loads(candidate)
        except Exception:
            pass
    return None


def repair_json(text):
    text = re.sub(r'("quote":\s*"[^"]*?)\n', r'\1"\n', text)
    text = re.sub(r",\s*}", "}", text)
    text = re.sub(r",\s*]", "]", text)
    text = re.sub(r'(\{|,)\s*(\w+)\s*:', r'\1"\2":', text)
    if text.count("{") > text.count("}"):
        text += "}" * (text.count("{") - text.count("}"))
    return text


def get_field(fields, name, key="value"):
    if not fields or name not in fields:
        return None
    v = fields[name]
    if isinstance(v, dict):
        v = v.get(key)
    if v is None or str(v).strip().lower() in ("null", "none", ""):
        return None
    return str(v).strip()


# ───────────────────────── 3. Compare ─────────────────────────
def _to_amount(s):
    s = re.sub(r"(?i)rs\.?|inr|₹|/-", "", s).replace(",", "")
    m = re.search(r"\d+(?:\.\d+)?", s)
    return float(m.group()) if m else None


def _to_date(s):
    for fmt in ("%d-%m-%Y", "%d/%m/%Y", "%d.%m.%Y", "%Y-%m-%d", "%d-%m-%y", "%d/%m/%y",
                "%d %B %Y", "%d %b %Y", "%d-%b-%Y", "%B %d, %Y", "%b %d, %Y"):
        try:
            return datetime.datetime.strptime(s.strip(), fmt).date()
        except ValueError:
            pass
    return None


def _words(s):
    return re.findall(r"[a-z0-9]+", s.lower())


def rule_match(v1, v2, field):
    """Exact, safe Python rules. Numbers and dates are ONLY ever checked here."""
    a, b = v1.lower().strip(), v2.lower().strip()
    if a == b:
        return True
    if field == "total_amount":
        x, y = _to_amount(a), _to_amount(b)
        return x is not None and x == y
    if field == "age":
        x, y = re.search(r"\d+", a), re.search(r"\d+", b)
        return bool(x and y and x.group() == y.group())
    if field == "gender":
        g = lambda s: "f" if s.startswith("f") else ("m" if s.startswith("m") else s)
        return g(a) == g(b)
    if field in ("admission_date", "discharge_date"):
        x, y = _to_date(a), _to_date(b)
        if x and y:
            return x == y
        return re.sub(r"[/\-.\s]", "-", a) == re.sub(r"[/\-.\s]", "-", b)
    w1, w2 = _words(a), _words(b)
    if field == "patient_name":
        titles = {"mr", "mrs", "ms", "smt", "shri", "dr", "miss"}
        return {w for w in w1 if w not in titles} == {w for w in w2 if w not in titles}
    s1, s2 = set(w1), set(w2)
    return bool(s1 and s2) and (s1 <= s2 or s2 <= s1)


def slm_same_meaning(field, v1, v2, q1=None, q2=None):
    """Ask the SLM whether two differently-worded values mean the same thing.
    Returns (same: bool|None, reason: str)."""
    label = field.replace("_", " ")
    prompt = (f"A claim form and a discharge summary describe the same hospital stay.\n"
              f"Claim form {label}: \"{v1}\"" + (f" (text: \"{q1}\")" if q1 else "") + "\n"
              f"Discharge summary {label}: \"{v2}\"" + (f" (text: \"{q2}\")" if q2 else "") + "\n\n"
              f"Do these refer to the SAME {label}, just written differently "
              f"(abbreviation, synonym, more detail)? Or are they DIFFERENT?\n"
              f"Answer with exactly one word, SAME or DIFFERENT, then ' - ' and one short reason.")
    raw = _generate([{"role": "system", "content": "You are a careful medical coding assistant."},
                     {"role": "user", "content": prompt}], max_new_tokens=60, use_adapter=False)
    head = raw.strip().upper()
    if head.startswith("SAME"):
        same = True
    elif head.startswith("DIFFERENT"):
        same = False
    else:
        same = None
    reason = raw.split("-", 1)[1].strip() if "-" in raw else raw.strip()
    return same, reason


# ───────────────────────── 2b. Quote checker (anti-hallucination) ─────────────────────────
import difflib

def _norm(s):
    return re.sub(r"\s+", " ", str(s).lower().replace(",", "")).strip()

def _fuzzy_in(needle, hay, threshold=0.85):
    """True if needle appears in hay, allowing small OCR differences."""
    n, h = _norm(needle), _norm(hay)
    if not n:
        return False
    if n in h:
        return True
    L = len(n)
    if L < 4:
        return False
    best = 0.0
    step = max(1, L // 4)
    for i in range(0, max(1, len(h) - L + 1), step):
        r = difflib.SequenceMatcher(None, n, h[i:i + L]).ratio()
        if r > best:
            best = r
            if best >= threshold:
                return True
    return False

def _number_in(value, text):
    """Amounts and ages: the same number must be written in the document."""
    v = _to_amount(str(value))
    if v is None:
        return False
    t = str(text).replace(",", "")
    for m in re.finditer(r"\d+(?:\.\d+)?", t):
        if float(m.group()) == v:
            return True
    return False

def _date_in(value, text):
    d = _to_date(str(value))
    for m in re.finditer(r"\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}|\d{1,2} [A-Za-z]{3,9} \d{4}", str(text)):
        if (d and _to_date(m.group()) == d) or _norm(m.group()) == _norm(value):
            return True
    return False

def verify_fields(fields, text):
    """Keep a value only if it can be found in the document text.
    Returns (clean_fields, rejected) where rejected maps field -> the value the model invented."""
    clean, rejected = {}, {}
    for f in CANONICAL_FIELDS:
        v, q = get_field(fields, f), get_field(fields, f, "quote")
        if v is None:
            clean[f] = {"value": None, "quote": None}
            continue
        if f in ("total_amount", "age"):
            ok = _number_in(v, text)
        elif f in ("admission_date", "discharge_date"):
            ok = _date_in(v, text)
        elif f == "gender":
            ok = bool(re.search(r"\b(male|female|m|f)\b", str(text).lower()))
        else:
            ok = _fuzzy_in(v, text) or (q is not None and _fuzzy_in(q, text) and _fuzzy_in(v, q, 0.6))
        if ok:
            clean[f] = {"value": v, "quote": q}
        else:
            clean[f] = {"value": None, "quote": None}
            rejected[f] = v
    return clean, rejected


def compare_fields(claim, disc, use_slm=True, rejected=None):
    """Each row: field, claim, discharge, status (match/meaning_match/different/missing/info/unclear), note."""
    rows = []
    for f in CANONICAL_FIELDS:
        c, d = get_field(claim, f), get_field(disc, f)
        row = {"field": f, "claim": c, "discharge": d}
        if f == "total_amount" and d is None:
            row.update(status="info", note="Amount appears only on the claim form")
        elif c is None and d is None:
            row.update(status="missing", note="Not found in either document")
        elif c is None or d is None:
            row.update(status="missing",
                       note=f"Not found in the {'claim form' if c is None else 'discharge summary'}")
        elif rule_match(c, d, f):
            row.update(status="match", note="")
        elif use_slm and f in MEANING_FIELDS:
            same, reason = slm_same_meaning(f, c, d, get_field(claim, f, "quote"),
                                            get_field(disc, f, "quote"))
            if same is True:
                row.update(status="meaning_match", note=f"Same meaning (AI): {reason}")
            elif same is False:
                row.update(status="different", note=f"Different (AI): {reason}")
            else:
                row.update(status="unclear", note=f"AI unsure: {reason}")
        else:
            row.update(status="different", note="Values do not match")
        for side, label in (("claim", "claim form"), ("disc", "discharge summary")):
            bad = (rejected or {}).get(side, {}).get(f)
            if bad is not None:
                extra = f"AI suggested '{bad}' for the {label}, but it does not appear in that document, so it was ignored."
                row["note"] = (row["note"] + " " + extra).strip()
        rows.append(row)
    return rows


def summarize(rows):
    count = lambda *s: sum(r["status"] in s for r in rows)
    return {"matches": count("match", "meaning_match"),
            "differences": count("different"),
            "needs_attention": count("missing", "unclear")}


# ───────────────────────── 4. Chat about the case ─────────────────────────
def chat(question, claim_text, disc_text, rows, history):
    findings = "\n".join(f"- {r['field']}: claim='{r['claim']}', discharge='{r['discharge']}', "
                         f"status={r['status']} {r['note']}" for r in rows)
    system = ("You are ClaimCheck Assistant, helping a human insurance reviewer understand ONE claim. "
              "Answer only from the two documents and findings below. If something is not in them, "
              "say clearly that it is not in the documents. When asked where something is, quote the exact "
              "line from the document text. If the findings and the document text disagree, trust the "
              "document text and say so. Explain medical terms simply when asked. "
              "Never approve, reject or give a final verdict: the human reviewer decides.\n\n"
              f"=== CLAIM FORM ===\n{claim_text}\n\n=== DISCHARGE SUMMARY ===\n{disc_text}\n\n"
              f"=== FIELD FINDINGS ===\n{findings}")
    msgs = [{"role": "system", "content": system}]
    msgs += history[-6:]
    msgs.append({"role": "user", "content": question})
    return _generate(msgs, max_new_tokens=350, use_adapter=False)


# ───────────────────────── 5. Audit log (hash chain) ─────────────────────────
def load_audit():
    if not os.path.exists(AUDIT_FILE):
        return []
    with open(AUDIT_FILE) as f:
        return [json.loads(l) for l in f if l.strip()]


def add_audit(case_id, claim_name, disc_name, summary, decision, reviewer, note):
    log = load_audit()
    prev = log[-1]["hash"] if log else "0" * 64
    entry = {"id": len(log) + 1, "case_id": case_id,
             "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
             "claim_file": claim_name, "discharge_file": disc_name,
             "ai_findings": summary, "human_decision": decision,
             "reviewer": reviewer, "note": note, "prev_hash": prev}
    entry["hash"] = hashlib.sha256((json.dumps(entry, sort_keys=True) + prev).encode()).hexdigest()
    with open(AUDIT_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")
    return entry


def verify_audit():
    """True if nobody edited the log (each hash still matches)."""
    prev = "0" * 64
    for e in load_audit():
        body = {k: v for k, v in e.items() if k != "hash"}
        if body["prev_hash"] != prev or hashlib.sha256(
                (json.dumps(body, sort_keys=True) + prev).encode()).hexdigest() != e["hash"]:
            return False
        prev = e["hash"]
    return True


In [ ]:
%%writefile app.py
import os, uuid, tempfile
import streamlit as st
import pandas as pd
import pipeline as P

st.set_page_config(page_title="ClaimCheck", page_icon="🏥", layout="wide")

ADAPTER_DIR = os.environ.get("ADAPTER_DIR", "./claimcheck-adapter")
STATUS_ICON = {"match": "✅ Match", "meaning_match": "🟢 Same meaning", "different": "❌ Different",
               "missing": "⚠️ Missing", "unclear": "❓ Unclear", "info": "ℹ️ Info"}


@st.cache_resource(show_spinner="Loading Qwen2.5-3B + ClaimCheck adapter (first time only, ~2 min)...")
def get_model():
    return P.load_model(ADAPTER_DIR)


get_model()
S = st.session_state
S.setdefault("stage", "input")      # input -> review -> done
S.setdefault("chat", [])


def reset():
    for k in list(S.keys()):
        del S[k]


def save_upload(f):
    path = os.path.join(tempfile.gettempdir(), f"{uuid.uuid4().hex}_{f.name}")
    with open(path, "wb") as out:
        out.write(f.getbuffer())
    return path


# ───────── Header ─────────
st.title("🏥 ClaimCheck")
st.caption("Health insurance claim review assistant · Qwen2.5-3B-Instruct fine-tuned with QLoRA · "
           "Veltris × Woxsen University")

# ───────── Stage 1: input ─────────
if S.stage == "input":
    st.subheader("1. Add the two documents")
    tab_up, tab_paste = st.tabs(["📁 Upload files", "📝 Paste text"])
    with tab_up:
        c1, c2 = st.columns(2)
        up_claim = c1.file_uploader("Claim form (txt / pdf / image)",
                                    type=["txt", "pdf", "png", "jpg", "jpeg"], key="up_claim")
        up_disc = c2.file_uploader("Discharge summary (txt / pdf / image)",
                                   type=["txt", "pdf", "png", "jpg", "jpeg"], key="up_disc")
    with tab_paste:
        c1, c2 = st.columns(2)
        pa_claim = c1.text_area("Claim form text", height=260, key="pa_claim")
        pa_disc = c2.text_area("Discharge summary text", height=260, key="pa_disc")

    if st.button("🔍 Analyse claim", type="primary", width="stretch"):
        if up_claim and up_disc:
            claim_text, m1 = P.read_document(save_upload(up_claim))
            disc_text, m2 = P.read_document(save_upload(up_disc))
            names = (up_claim.name, up_disc.name)
        elif pa_claim.strip() and pa_disc.strip():
            claim_text, m1, disc_text, m2 = pa_claim, "pasted", pa_disc, "pasted"
            names = ("pasted text", "pasted text")
        else:
            st.warning("Please give both documents (upload both, or paste both).")
            st.stop()
        if not claim_text or not disc_text:
            st.error(f"Could not read a document. Claim: {m1}. Discharge: {m2}.")
            st.stop()

        with st.status("Analysing...", expanded=True) as status:
            st.write(f"Read claim form ({m1}, {len(claim_text)} chars) and discharge summary "
                     f"({m2}, {len(disc_text)} chars)")
            st.write("Extracting fields from the claim form with the fine-tuned SLM...")
            claim_f, claim_raw = P.extract_fields(claim_text, "Claim Form")
            st.write("Extracting fields from the discharge summary...")
            disc_f, disc_raw = P.extract_fields(disc_text, "Discharge Summary")
            if claim_f is None or disc_f is None:
                status.update(label="Extraction failed", state="error")
                st.error("The model's output could not be read as JSON.")
                st.code((claim_raw if claim_f is None else disc_raw)[:800])
                st.stop()
            st.write("Comparing fields (Python rules + SLM meaning check)...")
            claim_f, rej_c = P.verify_fields(claim_f, claim_text)
            disc_f, rej_d = P.verify_fields(disc_f, disc_text)
            if rej_c or rej_d:
                st.write(f"Quote check: ignored {len(rej_c) + len(rej_d)} value(s) not found in the documents")
            rows = P.compare_fields(claim_f, disc_f, rejected={"claim": rej_c, "disc": rej_d})
            status.update(label="Analysis complete", state="complete")

        S.update(stage="review", case_id=f"CC-{uuid.uuid4().hex[:6].upper()}",
                 claim_text=claim_text, disc_text=disc_text, names=names,
                 claim_f=claim_f, disc_f=disc_f, rows=rows, summary=P.summarize(rows), chat=[])
        st.rerun()

# ───────── Stage 2: review ─────────
elif S.stage == "review":
    sm = S.summary
    st.subheader(f"2. Review case {S.case_id}")
    st.info("ClaimCheck shows findings only. **You** make the final decision.")
    a, b, c = st.columns(3)
    a.metric("Fields that agree", sm["matches"])
    b.metric("Differences", sm["differences"])
    c.metric("Need attention", sm["needs_attention"])

    df = pd.DataFrame([{"Field": r["field"].replace("_", " ").title(),
                        "Claim form": r["claim"] or "-", "Discharge summary": r["discharge"] or "-",
                        "Finding": STATUS_ICON.get(r["status"], r["status"]),
                        "Note": r["note"]} for r in S.rows])
    st.table(df.set_index("Field"))   # full text, wrapped: reviewers must read every note

    with st.expander("📄 View the original documents"):
        c1, c2 = st.columns(2)
        c1.text_area("Claim form", S.claim_text, height=300, disabled=True)
        c2.text_area("Discharge summary", S.disc_text, height=300, disabled=True)

    st.subheader("💬 Ask about this case")
    for m in S.chat:
        st.chat_message(m["role"]).write(m["content"])
    q = st.chat_input("e.g. What procedure was done? Explain the diagnosis simply.")
    if q:
        st.chat_message("user").write(q)
        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):
                ans = P.chat(q, S.claim_text, S.disc_text, S.rows, S.chat)
            st.write(ans)
        S.chat += [{"role": "user", "content": q}, {"role": "assistant", "content": ans}]

    st.divider()
    st.subheader("3. Reviewer decision")
    reviewer = st.text_input("Reviewer name", key="reviewer")
    note = st.text_area("Note (required for Reject / Send back)", key="note", height=80)
    b1, b2, b3 = st.columns(3)
    decision = None
    if b1.button("✅ Approve", type="primary", width="stretch"):
        decision = "APPROVED"
    if b2.button("❌ Reject", width="stretch"):
        decision = "REJECTED"
    if b3.button("↩️ Send back for more info", width="stretch"):
        decision = "SENT BACK"
    if decision:
        if not reviewer.strip():
            st.warning("Please enter the reviewer name.")
        elif decision != "APPROVED" and not note.strip():
            st.warning("Please add a note explaining the decision.")
        else:
            S.entry = P.add_audit(S.case_id, *S.names, sm, decision, reviewer.strip(), note.strip())
            S.stage = "done"
            st.rerun()

# ───────── Stage 3: done ─────────
elif S.stage == "done":
    e = S.entry
    st.balloons() if e["human_decision"] == "APPROVED" else None
    st.success(f"### Review complete\nCase **{e['case_id']}** marked **{e['human_decision']}** "
               f"by **{e['reviewer']}** at {e['timestamp'].replace('T', ' ')}.")
    st.caption(f"Audit record #{e['id']} · hash `{e['hash'][:16]}...`")
    if st.button("➕ Start next review", type="primary"):
        reset()
        st.rerun()

# ───────── Audit log (always available) ─────────
with st.sidebar:
    st.header("📜 Audit log")
    log = P.load_audit()
    if log:
        st.caption("✅ Log intact (hash chain verified)" if P.verify_audit()
                   else "⚠️ Log has been tampered with")
        st.dataframe(pd.DataFrame([{"#": x["id"], "Case": x["case_id"],
                                    "Decision": x["human_decision"], "By": x["reviewer"],
                                    "When": x["timestamp"][5:16].replace("T", " ")} for x in log]),
                     hide_index=True)
    else:
        st.caption("No reviews yet.")


In [ ]:
# ── Step 5: Start the app + public link ──────────────────────
import subprocess, time, re, os

subprocess.run("pkill -f 'streamlit run' ; pkill -f cloudflared", shell=True)

# Calm, professional light theme (no neon, no dark mode)
os.makedirs(".streamlit", exist_ok=True)
open(".streamlit/config.toml", "w").write(
    '[theme]\nbase = "light"\nprimaryColor = "#1b2a49"\nbackgroundColor = "#ffffff"\n'
    'secondaryBackgroundColor = "#f1f4f8"\ntextColor = "#1f2937"\nfont = "sans serif"\n'
    '[client]\ntoolbarMode = "viewer"\n')
st_proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--browser.gatherUsageStats", "false"],
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
time.sleep(5)

cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                           stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        url = m.group(); break
    time.sleep(1)

if url:
    print("🚀 ClaimCheck is live at:", url)
    print("   First open takes ~1-2 min while the model loads. Then it is fast.")
    print("   Keep this Colab tab open. If the link stops, rerun this cell.")
else:
    print("❌ No link yet. Tunnel log:"); print(open("cloudflared.log").read()[-2000:])

In [ ]:
# ── Step 6 (only if something breaks): show app logs ─────────
print(open("streamlit.log").read()[-3000:])